In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
%matplotlib inline
import tensorflow as tf
from tensorflow import keras
from keras.optimizers import Adam
from keras.losses import Loss
from joblib import Parallel, delayed
from scipy.optimize import minimize
from tensorflow.keras.initializers import GlorotUniform, GlorotNormal, HeUniform
import keras.backend as K
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
# Define DPD custom loss function as a class
class DPDLoss(Loss):
    def __init__(self, sigma, alpha):
        super().__init__()
        self.alpha = alpha
        self.sigma = float(sigma)

    def call(self, y_true, y_pred):
        diff = (y_true - y_pred)/self.sigma
        diff_sq = tf.clip_by_value(diff**2, 1e-10, 1e6)  # Prevents extreme values
        normal_dist = tf.exp(-0.5 * diff_sq)
        normal_dist = tf.clip_by_value(normal_dist, 1e-10, 1.0)  # Keep in safe range
        loss = 1/((self.sigma**self.alpha)*((1+self.alpha)**0.5)) - (1+1/self.alpha)*tf.reduce_mean((normal_dist/self.sigma)**self.alpha)
        # Debugging
        #tf.print("Loss:", loss)
        #loss = tf.debugging.check_numerics(loss, "NaN detected in loss function!")
        return loss

class HuberLoss(Loss):
    def __init__(self, delta=1.345, sig=None):
        super().__init__()
        self.delta = delta
        self.sig = sig

    def call(self, y_true, y_pred):
        error = (y_true - y_pred)/(self.sig + K.epsilon())
        # Huber loss computation
        is_small_error = tf.abs(error) <= self.delta
        squared_loss = 0.5 * tf.square(error)
        linear_loss = self.delta * (tf.abs(error) - 0.5 * self.delta)
        loss = tf.where(is_small_error, squared_loss, linear_loss)
        return tf.reduce_mean(loss)
        
class TukeyLoss(Loss):
    def __init__(self, delta=4.685, sig=None):
        super().__init__()
        self.delta = delta
        self.sig = sig

    def call(self, y_true, y_pred):
        error = (y_true - y_pred)/(self.sig + K.epsilon())
        # Apply Tukey's biweight function
        mask = tf.abs(error) <= self.delta
        loss = tf.where(mask, (self.delta**2 / 6) * (1 - (1 - (error / self.delta) ** 2) ** 3), self.delta**2 / 6)
        return tf.reduce_mean(loss)  # Return mean loss over batch

class LMLSLoss(Loss):
    def __init__(self):
        super().__init__()

    def call(self, y_true, y_pred):
        r = y_pred - y_true  # Compute residual error
        loss = tf.math.log(1 + (r**2) / 2)  # Apply LMLS formula
        return tf.reduce_mean(loss)  # Take mean loss over batch

class LTSLoss(Loss):
    def __init__(self, h):
        super().__init__()
        self.h = h

    def call(self, y_true, y_pred):
        residuals = tf.square(y_true - y_pred)  # Compute squared residuals
        sorted_residuals = tf.sort(residuals)   # Sort in ascending order
        #num_elements = tf.size(residuals)
        #k = tf.minimum(self.h, num_elements)
        k = self.h
        trimmed_residuals = sorted_residuals[:k]  # Keep only k smallest residuals
        return tf.reduce_mean(trimmed_residuals)  # Compute mean of the kept residuals

class LTALoss(Loss):
    def __init__(self, h):
        super().__init__()
        self.h = h

    def call(self, y_true, y_pred):
        residuals = tf.abs(y_true - y_pred)  # Compute absolute residuals
        sorted_residuals = tf.sort(residuals)   # Sort in ascending order
        #num_elements = tf.size(residuals)
        #k = tf.minimum(self.h, num_elements)
        k = self.h
        trimmed_residuals = sorted_residuals[:k]  # Keep only k smallest residuals
        return tf.reduce_mean(trimmed_residuals)  # Compute mean of the kept residuals

def trimmed_mean(arr, trim_fraction):
    arr_sorted = np.sort(arr)  # Step 1: Sort array
    trim_count = int(len(arr) * trim_fraction)  # Step 2: Compute number of elements to trim
    
    if trim_count == 0:  # Ensure at least one element is trimmed if possible
        return np.mean(arr)
    
    trimmed_arr = arr_sorted[:-trim_count]  # Step 3: Remove largest 20%
    return np.mean(trimmed_arr)  # Step 4: Compute mean of remaining elements

def H(sigma, alpha, diff):
    normal_dist = np.exp(-0.5 * (diff / sigma) ** 2) / sigma
    loss = 1 / ((sigma ** alpha) * np.sqrt(1 + alpha)) - (1 + 1 / alpha) * np.mean(normal_dist ** alpha)
    return loss

def sig_hat_MAD(resi):
    return 1.4826 * np.median(np.abs(resi - np.median(resi)))

In [ ]:
df = pd.read_csv('Concrete-scaled.csv')
print(df.shape)

In [ ]:
X = df.iloc[:,:-1]
Y = df['Y']
n = X.shape[0]
X.shape, Y.shape, n

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
def get_model():
    model = keras.Sequential([
        keras.layers.Dense(21, activation='relu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
        ])
    return model

# MLE, LTA, LTS, Huber, Tukey, LMLS

In [ ]:
def TMSE_comp_fold(train_idx, val_idx):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]  # Convert to NumPy array
    Y_train, Y_val = Y.iloc[train_idx], Y.iloc[val_idx]  # Convert to NumPy array
    
    n_epochs = 3000
    #### MLE
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
    # Predict on validation set
    mle_pred = model.predict(X_val,verbose=0).flatten()
    # Compute squared errors
    diff = Y_val - mle_pred
    tmse = [trimmed_mean(diff**2, trim_fraction=0.20)]
    
    ## scale related estimates used for other methods
    sig0 = sig_hat_MAD(diff)
    abs_diff = np.abs(diff); h = sum(abs_diff < 3*sig_hat_MAD(abs_diff))
    
    #### LTA (Adaptive)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=LTALoss(h = h))
    model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
    # Predict on validation set
    Y_pred = model.predict(X_val,verbose=0).flatten()
    # Compute squared errors
    diff = Y_val - Y_pred
    tmse_lta_ad = [trimmed_mean(diff**2, trim_fraction=0.20)]
    
    #### LTS (Adaptive)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=LTSLoss(h = h))
    model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
    # Predict on validation set
    Y_pred = model.predict(X_val,verbose=0).flatten()
    # Compute squared errors
    diff = Y_val - Y_pred
    tmse_lts_ad = [trimmed_mean(diff**2, trim_fraction=0.20)]
    
    #### Huber
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=HuberLoss(delta = 1.345, sig = sig0))
    model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
    # Predict on validation set
    Y_pred = model.predict(X_val,verbose=0).flatten()
    # Compute squared errors
    diff = Y_val - Y_pred
    tmse_huber = [trimmed_mean(diff**2, trim_fraction=0.20)]
    
    #### Tukey
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=TukeyLoss(delta = 4.685, sig = sig0))
    model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
    # Predict on validation set
    Y_pred = model.predict(X_val,verbose=0).flatten()
    # Compute squared errors
    diff = Y_val - Y_pred
    tmse_tukey = [trimmed_mean(diff**2, trim_fraction=0.20)]
    
    #### LMLS
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=LMLSLoss())
    model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
    # Predict on validation set
    Y_pred = model.predict(X_val,verbose=0).flatten()
    # Compute squared errors
    diff = Y_val - Y_pred
    tmse_lmls = [trimmed_mean(diff**2, trim_fraction=0.20)]
    
    return np.concatenate((tmse, tmse_lta_ad, tmse_lts_ad, tmse_huber, tmse_tukey, tmse_lmls))

In [ ]:
#TMSE_comp_fold(train_idx=[1,2],val_idx=[3,4])

In [ ]:
results = Parallel(n_jobs=-1)(delayed(TMSE_comp_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(X))
methods = ['MLE', 'LTA_Ad', 'LTS_Ad', 'Huber','Tukey','LMLS']
comp_cv = pd.DataFrame(np.mean(results, axis=0).reshape(-1,1), columns=['20% TMSE'],index=methods)
comp_cv['20% TMSE'] # 10-Fold CV-TMSE results for existing methods

# DPD

In [ ]:
al = np.array([0.1,0.3,0.5,0.7,1])

def TMSE_DPD_fold(train_idx, val_idx):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]  # Convert to NumPy array
    Y_train, Y_val = Y.iloc[train_idx], Y.iloc[val_idx] # Convert to NumPy array
    
    sig0 = 5.5 #1 # Set your initial sigma value
    
    num_iterations = 15
    n_epochs = 200
    
    tt = np.array([])
    for alpha in al:
        np.random.seed(42)
        tf.random.set_seed(42)
        model = get_model()
        v1 = 1; dv = 1
        i=0
        while (i < num_iterations and dv > 0.0001):
            model.compile(optimizer=Adam(learning_rate=0.001), loss=DPDLoss(sigma = sig0, alpha=alpha))
            if(i>0):
                model.set_weights(weights)
            model.fit(X_train, Y_train, epochs=n_epochs, verbose = 0)
            dpd_pred_train = model.predict(X_train, verbose=0).flatten()
            weights = model.get_weights()  #  print(weights)
            diff = np.array(Y_train - dpd_pred_train)
            sig0 = minimize(H, sig0, args = (alpha,diff), method='L-BFGS-B', bounds = [(0.001,10)]).x[0]
            v0 = v1; v1 = H(sig0,alpha,diff); dv = abs(v0 - v1)
            i=i+1

        dpd_pred_val = model.predict(X_val,verbose=0).flatten()
        diff = Y_val - dpd_pred_val
        tmse = [trimmed_mean(diff**2, trim_fraction=0.20)]
        tt = np.concatenate((tt, tmse))
    
    return tt

In [ ]:
#TMSE_DPD_fold(train_idx=[1,2],val_idx=[3,4])

In [ ]:
results = Parallel(n_jobs=-1)(delayed(TMSE_DPD_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(X))
DPD_cv = pd.DataFrame(np.mean(results, axis=0).reshape(-1,1), columns=['20% TMSE'], index=al)
DPD_cv['20% TMSE'] # 10-Fold CV-TMSE results for the DPD-based learning